# Live Regional Macro Sentiment Polling

## Integration Guide: Pre-Aggregated Regional Macro Sentiment Index

This notebook demonstrates how to integrate the **Permutable AI Regional Macro Sentiment Index API** into an analytical workflow. The regional macro index endpoint returns sentiment pre-aggregated into hourly buckets at the **country** level, making it straightforward to build monitoring dashboards and research workflows without client-side aggregation.

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing in this notebook constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment indicators derived here reflect aggregated model outputs and should not be used as the sole basis for any investment decision.

### How the Regional Index Differs from the Asset Index

| | Asset Index (`/headlines/index/ticker/live`) | Regional Macro Index (`/index/macro/live/regional`) |
|---|---|---|
| **Dimension** | Asset ticker (e.g. `BZ_COM`) | Country (e.g. `united states`) |
| **Key metrics** | `sentiment_sum`, `sentiment_abs_sum`, `headline_count`, `sentiment_std` | `sentiment_avg`, `sentiment_sum`, `headline_count`, `sentiment_std` |
| **Scope** | Asset-specific news | Macro-economic topics by country |
| **Model** | Fixed asset model | `macro_1` / `macro_2` / `macro_3` |
| **Best for** | Asset monitoring, avg sentiment ratio | Country-level macro monitoring, geopolitical sentiment |

### What the Endpoint Returns

`GET /v1/index/macro/live/regional/{model_id}` returns the latest 2 hours of pre-aggregated regional sentiment. Each record maps to the `MacroRegionalSentiment` schema:

| Field | Type | Description |
|---|---|---|
| `publication_time` | `datetime` | Start or end of the hourly bucket |
| `topic_name` | `str` | Macro topic (e.g. `Economic Data`, `Monetary Policy`) |
| `country` | `str` | Country name (e.g. `united states`) |
| `headline_count` | `int` | Number of headlines in this bucket |
| `sentiment_avg` | `float` | Mean sentiment score across headlines in this bucket (−1 to +1) |
| `sentiment_sum` | `float` | Sum of sentiment scores |
| `sentiment_std` | `float` | Standard deviation of sentiment (dispersion / disagreement) |

**Primary signal metric — `sentiment_avg`:**

```
sentiment_avg ∈ [−1, +1]
```

Values near +1 indicate strong bullish macro sentiment for the country; near −1 indicate bearish. Unlike the asset index, `sentiment_avg` is already normalised so no further division is needed.

### Polling Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Every 15 minutes                                                       │
│                                                                         │
│  [API] ──► fetch_live_regional(model_id)                                │
│                  │                                                      │
│                  ▼                                                      │
│          upsert_regional()  ──► [SQLite: macro_regional_sentiment]      │
│                                         │                               │
│                                         ▼                               │
│              load from DB ──► rolling sentiment indicator per country   │
└─────────────────────────────────────────────────────────────────────────┘
```

**Upsert key:** `(publication_time, topic_name, country, index_type)`. The live endpoint always returns the current 2-hour window, so successive polls overlap. Upserting on this composite key keeps the database deduplicated without bookkeeping.

### Pipeline Steps

1. **Install** dependencies and configure API credentials
2. **Define** the fetch and upsert functions
3. *(Optional)* **Historical backfill** — populate the database with up to 90 days of history before starting live polling. Recommended for monitoring dashboards that need trend charts from the first run.
4. **Dry-run** a single poll to validate connectivity and inspect the regional schema
5. **Visualise** the initial hourly snapshot (world map, country time series, sentiment heatmap)
6. **Run** the 15-minute polling loop
7. **Visualise** the accumulated live window (country lines, rolling heatmap, indicator heatmap)
8. **Derive** a rolling sentiment indicator (HIGH / NEUTRAL / LOW) with configurable thresholds

### Prerequisites

- A Permutable AI API key (available from your account dashboard)
- Python 3.9+ with the packages listed in the cell below

In [ ]:
# Install required packages — run this cell once, then restart the kernel if needed
%pip install requests pandas matplotlib seaborn plotly

In [ ]:
import getpass
import sqlite3
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

sys.path.insert(0, str(Path("../").resolve()))
from utils.plotting import (
    plot_sentiment_map,
    plot_sentiment_heatmap,
    plot_sentiment_table,
    plot_sentiment_time_series,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

In [ ]:
# ── API credentials ────────────────────────────────────────────────────────────
# The key is read via getpass so it is never displayed or stored in the notebook
API_KEY = getpass.getpass("Enter your Permutable AI API key: ")

In [ ]:
# ── Endpoint configuration ─────────────────────────────────────────────────────
BASE_URL = "https://copilot-api.permutable.ai/v1"

# Model ID: macro_1 | macro_2 | macro_3
MODEL_ID = "macro_1"

# Country preset — filters which countries are returned. Options include: G7, ALL,
# or an individual country name (e.g. "united states"). Check the API docs for full list.
COUNTRY_PRESET = "ALL"

# Topic preset — "ALL" returns all macro topics; or specify a topic name (e.g. "Economic Data")
TOPIC_PRESET = "ALL"

# Language and source filters (use "ALL" to include all sources)
LANGUAGE_PRESET = "ALL"
SOURCE_PRESET = "ALL"
SOURCE_COUNTRY_PRESET = "ALL"

# Index type: COMBINED = EXPLICIT + IMPLICIT; EXPLICIT = direct country mention; IMPLICIT = inferred
INDEX_TYPE = "COMBINED"  # EXPLICIT | IMPLICIT | COMBINED

# sparse=True returns only buckets that have headlines (recommended)
# align_to_period_end=True timestamps each bucket at the close of the hour
SPARSE = True
ALIGN_TO_PERIOD_END = True

# ── Polling interval ───────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 15 * 60  # poll every 15 minutes

ROLLING_WINDOW = 5  # hours

# ── Local storage ──────────────────────────────────────────────────────────────
DB_PATH = Path("macro_regional_sentiment.db")

print(f"Model ID         : {MODEL_ID}")
print(f"Country preset   : {COUNTRY_PRESET}")
print(f"Topic preset     : {TOPIC_PRESET}")
print(f"Index type       : {INDEX_TYPE}")
print(f"Polling interval : {POLL_INTERVAL_SECONDS // 60} minutes")
print(f"Local database   : {DB_PATH.resolve()}")

In [ ]:
def fetch_live_regional(model_id: str = MODEL_ID) -> list[dict]:
    """
    Fetch the live regional macro sentiment index.

    Calls GET /v1/index/macro/live/regional/{model_id} and returns a flat list
    of record dicts matching the MacroRegionalSentiment schema. A single API call
    returns data for ALL countries in COUNTRY_PRESET — no per-country looping needed.
    A 'fetched_at' timestamp is appended so you can track when each record was retrieved.
    """
    url = f"{BASE_URL}/headlines/index/macro/live/regional/{model_id}"
    params = {
        "country_preset": COUNTRY_PRESET,
        "topic_preset": TOPIC_PRESET,
        "language_preset": LANGUAGE_PRESET,
        "source_preset": SOURCE_PRESET,
        "source_country_preset": SOURCE_COUNTRY_PRESET,
        "index_type": INDEX_TYPE,
        "sparse": str(SPARSE).lower(),
        "align_to_period_end": str(ALIGN_TO_PERIOD_END).lower(),
    }
    headers = {"x-api-key": API_KEY}

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    records = response.json()
    ts = datetime.now(timezone.utc).isoformat()
    for r in records:
        r["fetched_at"] = ts
        r["index_type"] = INDEX_TYPE
    return records


# Quick connectivity check
print(f"Testing connectivity for model {MODEL_ID}...")
test_records = fetch_live_regional()
countries_seen = len({r["country"] for r in test_records})
print(f"  OK — {len(test_records)} records returned across {countries_seen} countries")

In [ ]:
def get_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def setup_database() -> None:
    """Create the macro_regional_sentiment table and index if they do not already exist."""
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS macro_regional_sentiment (
                publication_time  TEXT NOT NULL,
                topic_name        TEXT NOT NULL,
                country           TEXT NOT NULL,
                index_type        TEXT NOT NULL,
                headline_count    INTEGER,
                sentiment_avg     REAL,
                sentiment_sum     REAL,
                sentiment_std     REAL,
                fetched_at        TEXT,
                PRIMARY KEY (publication_time, topic_name, country, index_type)
            )
        """)
        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_mrs_country_time
            ON macro_regional_sentiment (country, publication_time)
        """)
    print(f"Database ready: {DB_PATH.resolve()}")


def upsert_regional(records: list[dict]) -> int:
    """
    Upsert regional macro sentiment records into the local SQLite database.

    INSERT OR REPLACE ensures that re-polling the same 2-hour window does not
    create duplicate rows. Records are uniquely identified by the composite key
    (publication_time, topic_name, country, index_type).

    Returns the number of rows written.
    """
    if not records:
        return 0

    columns = [
        "publication_time",
        "topic_name",
        "country",
        "index_type",
        "headline_count",
        "sentiment_avg",
        "sentiment_sum",
        "sentiment_std",
        "fetched_at",
    ]
    placeholders = ", ".join(["?"] * len(columns))
    sql = f"INSERT OR REPLACE INTO macro_regional_sentiment ({', '.join(columns)}) " f"VALUES ({placeholders})"
    rows = [tuple(r.get(c) for c in columns) for r in records]

    with get_connection() as conn:
        conn.executemany(sql, rows)
    return len(rows)


setup_database()
print("upsert_regional() defined")

## Historical Backfill *(Optional — Recommended for Dashboards)*

> **Skip this section if you only want live data.** Run it before the dry-run if you want your local database pre-populated with historical data so charts are meaningful from the first run.

The historical endpoint (`GET /v1/index/macro/historical/regional/{model_id}`) supports up to 90 days of lookback and uses keyset pagination — each response contains `has_more` and `next_token`.

**Live vs historical endpoint comparison:**

| | Live (`/live/regional/{model_id}`) | Historical (`/historical/regional/{model_id}`) |
|---|---|---|
| **Window** | Last 2 hours | `start_date` to `end_date` (up to 90 days) |
| **Pagination** | None (single response) | Keyset pagination via `next_token` |
| **Use case** | Continuous polling loop | Seeding database with history |

### Why backfill?

Without backfill, a fresh installation shows only the last 2 hours of data. Running the backfill once seeds 7 days (or more) of history so your monitoring dashboard has context from the first run.

### Companion Application

For production use, see [`app/live_regional_polling`](../../../app/live_regional_polling), which runs this backfill automatically on first startup and then polls every 15 minutes indefinitely.

In [ ]:
from datetime import timedelta

BACKFILL_DAYS = 7  # days of history to fetch; max 90


def fetch_historical_regional(
    model_id: str,
    start_date,
    end_date=None,
) -> list[dict]:
    """
    Paginate through GET /v1/index/macro/historical/regional/{model_id} and return
    all matching records as a flat list.

    Keyset pagination: each response includes 'has_more' and 'next_token'.
    Loops until has_more is False, passing next_token into every subsequent request.
    """
    url = f"{BASE_URL}/headlines/index/macro/historical/regional/{model_id}"
    params = {
        "start_date": start_date.isoformat(),
        "country_preset": COUNTRY_PRESET,
        "topic_preset": TOPIC_PRESET,
        "language_preset": LANGUAGE_PRESET,
        "source_preset": SOURCE_PRESET,
        "source_country_preset": SOURCE_COUNTRY_PRESET,
        "index_type": INDEX_TYPE,
        "sparse": str(SPARSE).lower(),
        "align_to_period_end": str(ALIGN_TO_PERIOD_END).lower(),
        "limit": 1000,
    }
    if end_date:
        params["end_date"] = end_date.isoformat()
    headers = {"x-api-key": API_KEY}

    all_records, page = [], 1
    while True:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        body = response.json()
        records = body["data"]
        ts = datetime.now(timezone.utc).isoformat()
        for r in records:
            r["fetched_at"] = ts
            r["index_type"] = INDEX_TYPE
        all_records.extend(records)
        print(f"    page {page:3d}: {len(records):5d} records  |  total so far: {len(all_records):6d}")
        if not body["has_more"]:
            break
        params["next_token"] = body["next_token"]
        page += 1
    return all_records


def backfill(days_back: int = BACKFILL_DAYS) -> None:
    """Fetch and upsert historical regional macro sentiment."""
    start = (datetime.utcnow() - timedelta(days=days_back)).date()
    print(f"Backfilling model {MODEL_ID} from {start} ({days_back} days)\n")
    try:
        records = fetch_historical_regional(MODEL_ID, start)
        n = upsert_regional(records)
        print(f"  {n} rows upserted into macro_regional_sentiment")
    except requests.HTTPError as e:
        print(f"  HTTP {e.response.status_code} — {e.response.text[:200]}")
    except Exception as e:
        print(f"  {type(e).__name__}: {e}")


# ── Run backfill ───────────────────────────────────────────────────────────────
# Comment out the line below to skip the backfill and proceed directly to the dry run.
backfill()

## Dry Run

Fetch and store regional sentiment data once. Use this to verify your API key and inspect the raw schema before starting the polling loop.

In [ ]:
print(f"Running dry-run poll for model {MODEL_ID}...\n")

try:
    records = fetch_live_regional()
    n_written = upsert_regional(records)
    countries_seen = len({r["country"] for r in records})
    topics_seen = len({r["topic_name"] for r in records})
    print(f"  {len(records):4d} records fetched | {n_written:4d} rows written")
    print(f"  Countries: {countries_seen}  |  Topics: {topics_seen}")
except requests.HTTPError as e:
    print(f"  HTTP {e.response.status_code} — {e.response.text[:200]}")
except Exception as e:
    print(f"  {type(e).__name__}: {e}")

print()

# Preview stored data
with get_connection() as conn:
    sample = pd.read_sql(
        "SELECT * FROM macro_regional_sentiment ORDER BY publication_time DESC LIMIT 10",
        conn,
    )
print("Sample rows from database:")
display(sample)

## Initial Visualisation

Inspect the first hourly snapshot:

1. **World choropleth map** — mean `sentiment_avg` per country for the latest available hour
2. **Country time series** — `sentiment_avg` over time for the top countries by headline volume
3. **5-hour rolling sentiment heatmap** — country × hour with date-boundary ticks

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM macro_regional_sentiment", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)

if df.empty:
    print("No data yet — run the dry-run cell above first.")
else:
    # Aggregate to (country, hour) across topics
    hourly = (
        df.groupby(["country", "publication_time"])
        .agg(
            sentiment_avg=("sentiment_avg", "mean"),
            sentiment_sum=("sentiment_sum", "sum"),
            headline_count=("headline_count", "sum"),
            sentiment_std=("sentiment_std", "mean"),
        )
        .reset_index()
        .sort_values("publication_time")
    )

    # Top countries by total headline volume
    top_countries = hourly.groupby("country")["headline_count"].sum().nlargest(16).index.tolist()
    colors = sns.color_palette("muted", len(top_countries))

    # ── Plot 1: Avg sentiment choropleth map — period + topic dropdowns ─────
    plot_sentiment_map(df).show()

    # ── Plot 2: Sentiment avg + headline count — country + topic dropdowns ──
    plot_sentiment_time_series(df, hourly, top_countries).show()

    print(f"Total rows in database : {len(df)}")
    print(f"Countries              : {df['country'].nunique()}")
    print(f"Topics                 : {df['topic_name'].nunique()}")

## 15-Minute Polling Loop

This cell runs indefinitely, polling the regional macro index every 15 minutes and upserting results into the local SQLite database.

**Stop:** press the ■ (Interrupt Kernel) button or use **Kernel → Interrupt**.

In production this logic runs as a scheduled container service. See [`app/live_regional_polling`](../../../app/live_regional_polling) for the production-ready Docker Compose implementation.

In [ ]:
poll_count = 0
print(f"Starting polling loop — model {MODEL_ID}, country preset {COUNTRY_PRESET}")
print(f"Polling every {POLL_INTERVAL_SECONDS // 60} minutes. Interrupt kernel to stop.\n")

try:
    while True:
        poll_count += 1
        t_start = datetime.now(timezone.utc)
        print(f"[Poll {poll_count}]  {t_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")

        try:
            records = fetch_live_regional()
            n = upsert_regional(records)
            countries_in_poll = len({r["country"] for r in records})
            print(f"  {len(records):4d} records | {n:4d} upserted | {countries_in_poll} countries")
        except requests.HTTPError as e:
            print(f"  HTTP {e.response.status_code} — skipping, will retry next poll")
        except Exception as e:
            print(f"  {type(e).__name__}: {e}")

        elapsed = (datetime.now(timezone.utc) - t_start).total_seconds()
        wait = max(0.0, POLL_INTERVAL_SECONDS - elapsed)
        print(f"  Completed in {elapsed:.1f}s.  Next poll in {wait / 60:.1f} min.\n")
        time.sleep(wait)

except KeyboardInterrupt:
    print(f"\nPolling stopped after {poll_count} poll(s).")
    print(f"All data saved to: {DB_PATH.resolve()}")

## Post-Poll Visualisation

Run this cell after one or more polling cycles to explore the accumulated data:

1. `sentiment_avg` over time per country with ±1 std deviation band
2. Headline count (news flow) per country — hourly time series
3. 5-hour rolling sentiment heatmap — country × hour

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM macro_regional_sentiment", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)
df = df.sort_values("publication_time")

if df.empty:
    print("No data yet — run the polling loop above first.")
else:
    ts = (
        df.groupby(["country", "publication_time"])
        .agg(
            sentiment_avg=("sentiment_avg", "mean"),
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_std=("sentiment_std", "mean"),
            headline_count=("headline_count", "sum"),
        )
        .reset_index()
        .sort_values("publication_time")
    )

    top_countries = ts.groupby("country")["headline_count"].sum().nlargest(16).index.tolist()
    colors = sns.color_palette("muted", len(top_countries))

    # ── Plot 1: Avg sentiment choropleth map — period + topic dropdowns ─────
    plot_sentiment_map(df).show()

    # ── Plot 2: Sentiment avg + headline count — country + topic dropdowns ──
    plot_sentiment_time_series(df, ts, top_countries).show()

    # ── Plot 3: 5-hour rolling avg sentiment heatmap — topic dropdown ────────
    plot_sentiment_heatmap(df, top_countries, rolling_window=ROLLING_WINDOW).show()

## Sentiment Aggregation Example

Demonstrates how to load the accumulated regional sentiment from the local database and derive a rolling indicator. This example is intended for **research and monitoring purposes only** and does not constitute financial advice.

> **Note — Simplified Baseline:** The aggregation shown here is intentionally straightforward: `sentiment_avg` values are averaged uniformly across all topics and rolled into a single country-level indicator with equal weighting. This provides a useful starting point for monitoring, but treats every topic and every `index_type` as equally informative regardless of country context.
>
> A more sophisticated production workflow would replace this uniform average with **ML/modelling techniques** to extract a richer, strategy-ready signal:
>
> - **Feature engineering across topics** — rather than averaging uniformly, learn a weight per topic (e.g. `Monetary Policy`, `Economic Data`, `Geopolitical Risk`) that reflects its historical predictive power for each country's macro regime. Topics that have consistently led realised macro outcomes receive higher weight; noisy or lagging topics are discounted.
> - **Domestic vs. international weighting (`index_type`)** — the `index_type` field distinguishes domestic coverage of a country from international coverage of it. A model can learn that for some countries domestic sentiment leads (e.g. closed economies with state media), while for others the international narrative is more predictive — and calibrate the split accordingly per country.
> - **Country-specific model calibration** — rather than applying a one-size-fits-all threshold, fit separate feature weights per country. Sentiment dynamics differ materially across economies: `Monetary Policy` news carries different signal for the US than for an emerging market with a pegged currency.
> - **Supervised signal construction** — label historical windows with realised macro outcomes (e.g. FX returns, rate decision surprises, credit spread moves) and train a regression or classification model to learn which combination of topic weights and `index_type` splits best explains those outcomes.
>
> The output of such a pipeline is a **country-level composite score with learned, interpretable feature weights** — a more robust and actionable input to systematic macro strategies than a uniform rolling mean.

**Aggregation logic (baseline):** aggregate topic-level records to (country, hour), apply a rolling mean of `sentiment_avg`, then threshold to **High Positive / Neutral / High Negative** avg sentiment.

| Metric | Formula | Range |
|---|---|---|
| `sentiment_avg` | Mean sentiment across all headlines in the bucket | [−1, +1] |
| `headline_count` | Total headlines in the bucket | ≥ 0 |
| `sentiment_std` | Mean std dev across topics (dispersion) | ≥ 0 |

### Rolling Window

The smoothed sentiment uses a **5-hour rolling mean** (`window=5`) — each hour's value is the average of the current hour and the 4 preceding hours per country. This dampens short-lived spikes.

| `window` | Behaviour |
|---|---|
| 1 | No smoothing — raw hourly sentiment_avg |
| 3 | Light smoothing — responsive to intraday shifts |
| **5** | **Moderate smoothing (default)** |
| 12 | Heavy smoothing — half-day trends |
| 24 | Daily smoothing — suitable for multi-day analysis |

Adjust `ROLLING_WINDOW` in the code cell below to suit your use case. Similarly, adjust `UPPER_THRESHOLD` and `LOWER_THRESHOLD` (defaults ±0.15). Countries above the upper threshold are labelled **High Positive avg sentiment**; below the lower threshold, **High Negative avg sentiment**; within the band, **Neutral**.

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM macro_regional_sentiment", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)

# ── 1. Aggregate to (country, hour) ───────────────────────────────────────────
df_agg = (
    df.groupby(["country", "publication_time"])
    .agg(
        sentiment_avg=("sentiment_avg", "mean"),
        sentiment_sum=("sentiment_sum", "sum"),
        headline_count=("headline_count", "sum"),
        sentiment_std=("sentiment_std", "mean"),
    )
    .reset_index()
    .sort_values(["country", "publication_time"])
)

# ── 2. Rolling avg sentiment: sum(sentiment_sum) / sum(headline_count) ───────────
# Avoids mean-reversion artefacts — sustained directional coverage accumulates
# rather than cancelling, while still being normalised to [−1, +1].

df_agg["avg_sentiment"] = (
    df_agg.groupby("country")
    .apply(
        lambda g: (
            g["sentiment_sum"].rolling(ROLLING_WINDOW, min_periods=1).sum()
            / g["headline_count"].rolling(ROLLING_WINDOW, min_periods=1).sum()
        )
    )
    .reset_index(level=0, drop=True)
    .fillna(0)
)

# ── 3. Threshold-based sentiment indicator ────────────────────────────────────
UPPER_THRESHOLD = 0.15  # above this → High Positive avg sentiment
LOWER_THRESHOLD = -0.15  # below this → High Negative avg sentiment

df_agg["indicator"] = "Neutral"
df_agg.loc[df_agg["avg_sentiment"] >= UPPER_THRESHOLD, "indicator"] = "High Positive"
df_agg.loc[df_agg["avg_sentiment"] <= LOWER_THRESHOLD, "indicator"] = "High Negative"

# ── 4. Real-Time Avg Sentiment — 5-Hour Rolling ─────────────────────────────────────────────
plot_sentiment_table(
    df,
    rolling_window=ROLLING_WINDOW,
    upper_threshold=UPPER_THRESHOLD,
    lower_threshold=LOWER_THRESHOLD,
).show()

# ── 5. Avg sentiment heatmap — topic dropdown (Plotly) ──────────────────────────
top_countries = df_agg.groupby("country")["headline_count"].sum().nlargest(16).index.tolist()
if len(df_agg) > 1 and top_countries:
    plot_sentiment_heatmap(df, top_countries, rolling_window=ROLLING_WINDOW).show()

## Next Steps: Production Deployment

This notebook is an **integration reference** — it covers the complete workflow from API authentication through to a rolling sentiment indicator across countries.

For a production deployment, see the accompanying **Live Regional Polling Application** in [`app/live_regional_polling`](../../../app/live_regional_polling), which packages this workflow into three Docker services: a **poller**, a **REST API**, and a **monitoring dashboard** with a live world map choropleth.

### Related Resources

- [Permutable AI API Documentation](https://docs.permutable.ai)
- [`app/live_regional_polling`](../../../app/live_regional_polling) — production-ready containerised polling app for this notebook's workflow
- `notebooks/backtesting/regional_cross_country_signal_assessment.ipynb` — backtesting pipeline for historical analysis of regional macro sentiment across countries
- `notebooks/live/index_sentiment_polling.ipynb` (in `headline_asset_sentiment`) — equivalent guide for the asset-level sentiment index